In [ ]:
%load_ext autoreload
%autoreload 2

from scripts.trajectory import Trajectory, StepToolCall, ToolResultDescription, StepSystemPrompt, StepUser, StepAgent

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
dataset_bfcl = "../data/AgentProcessBench/bfcl.jsonl"
dataset_tau2 = "../data/AgentProcessBench/tau2.jsonl"
dataset = "../data/AgentProcessBench/hotpotqa.jsonl"

bfcl = Trajectory(dataset_bfcl)
tau2 = Trajectory(dataset_tau2)
hotpotqa = Trajectory(dataset)


# data = Trajectory(dataset)
# for step in data:
#     print(f"label={step.label} | reason={step.reason} | {step}")

    # if getattr(step, "label", None) == -1:
    #     print(
    #         f"broken at trajectory={step.trajectory_index}, label={step.label}, "
    #         f"message={step.message_index}\n"
    #         f"reason: {getattr(step, 'reason', 'Reason unavailable; rerun the Trajectory definition cell.')}"
    #     )

In [3]:
import pandas as pd

# Exploratory Data Analysis.

df_a = pd.DataFrame(bfcl)
df_b = pd.DataFrame(tau2)
df_c = pd.DataFrame(hotpotqa)

df = pd.concat(
    [df_a, df_b, df_c],
    ignore_index=True
)
df

,content,label,reason,tool_calls,trajectory_index,message_index,name,tool_call_id,inferred
0,You are a helpful assistant and an expert in f...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"<state_info class_name=""GorillaFileSystem"">\n{...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Hey, can you set up a new directory named 'Pro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,,1.0,Annotated +1: this assistant step is correct a...,"[{'id': 'call_multi_turn_base_10_0_0_0', 'type...",0.0,3.0,NaN,NaN,NaN
4,"{""current_working_directory"": ""/alex""}",NaN,NaN,NaN,0.0,4.0,pwd,call_multi_turn_base_10_0_0_0,False
...,...,...,...,...,...,...,...,...,...
16497,<think>We need to know genres of the two films...,-1.0,Annotated -1: The submitted answer 'No. Both f...,NaN,249.0,2.0,NaN,NaN,NaN
16498,No effective tool call detected. Please call a...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16499,,1.0,Annotated +1: this assistant step is correct a...,"[{'id': 'call_Nx4R5KW5GMYOReS68BPp3JC7', 'type...",249.0,4.0,NaN,NaN,NaN
16500,"{""result"": [[{""document"": {""id"": ""13305660"", ""...",NaN,NaN,NaN,249.0,5.0,search,call_Nx4R5KW5GMYOReS68BPp3JC7,False


In [4]:
# How imbalanced are -1 / 0 / +1?

label_counts = df["label"].value_counts()
label_pct = df["label"].value_counts(normalize=True).mul(100)

label_summary = pd.DataFrame({
    "count": label_counts,
    "percentage": label_pct
})

print("Imbalance of data set\n", label_summary, "\n")

Imbalance of data set
        count  percentage
label                   
 1.0    4735   68.812673
-1.0    1859   27.016422
 0.0     287    4.170905 



In [5]:
# Are most tool calls correct?

df["tool_name"] = df["tool_calls"].apply(
    lambda x: x[0]["schema"]["name"]
    if isinstance(x, list) and len(x) > 0
    else None
)
tool_df = df[df["tool_name"].notna()]

are_tools_correct = tool_df["label"].value_counts(normalize=True).mul(100)
print("Are most tool calls correct\n", are_tools_correct, "\n")

Are most tool calls correct
 label
 1.0    69.906496
-1.0    24.729331
 0.0     5.364173
Name: proportion, dtype: float64 



In [6]:
# Which tools produce the most errors?

tool_most_errors = df.groupby("tool_name")["label"].value_counts()

print("Tool most errors\n", tool_most_errors, "\n")

Tool most errors
 tool_name             label
activateParkingBrake   1.0      8
                       0.0      2
add_to_watchlist       1.0     17
                      -1.0      1
authenticate_travel    1.0      5
                               ..
view_messages_sent     1.0     19
                      -1.0      1
                       0.0      1
wc                     1.0     19
                      -1.0      7
Name: count, Length: 259, dtype: int64 



In [ ]:
# Are errors more common late in trajectories?

df["max_message_index"] = (
    df.groupby("trajectory_index")["message_index"]
      .transform("max")
)

df["relative_position"] = (
    df["message_index"] / df["max_message_index"]
)

df["trajectory_stage"] = pd.cut(
    df["relative_position"],
    bins=[0, 0.33, 0.66, 1],
    labels=["early", "middle", "late"],
    include_lowest=True
)

error_by_stage = (
    df.dropna(subset=["label"])
      .groupby("trajectory_stage", observed=True)["label"]
      .apply(lambda x: (x == -1).mean())
)

print("Error by stage \n", error_by_stage, "\n")

Error by stage 
 trajectory_stage
early     0.164586
middle    0.336138
late      0.376325
Name: label, dtype: float64 



In [8]:

# Are tool calls more likely to fail than normal messages?
df["is_tool_call"] = df["tool_name"].notna()
are_tool_calls_likely_to_fail = df.groupby("is_tool_call")["label"].apply(
    lambda x: (x == -1).mean()
)

print("Are tool calls more likely to fail than normal messages?\n", are_tool_calls_likely_to_fail, "\n")

Are tool calls more likely to fail than normal messages?
 is_tool_call
False    0.070889
True     0.225589
Name: label, dtype: float64 



In [9]:
# How long are good vs bad steps?
# Keep rows with non-empty content
text_df = df[
    df["content"].notna() &
    df["content"].ne("")
].copy()

# Create character length
text_df["char_length"] = text_df["content"].str.len()

# Compare text length for rows with vs without tool calls
text_length_by_tool_call = (
    text_df
    .groupby(text_df["tool_calls"].notna())["char_length"]
    .describe()
)

text_length_by_tool_call

,count,mean,std,min,25%,50%,75%,max
tool_calls,,,,,,,,
False,12043.0,1332.034045,4242.239267,1.0,69.0,249.0,677.0,32306.0
True,787.0,271.528590,308.256928,8.0,100.0,160.0,309.5,2560.0


In [10]:
import ast
import json
import numpy as np
import pandas as pd


def engineer_features(df: pd.DataFrame):
    """
    Convert AgentProcessBench dataframe into a modeling dataset.

    Returns
    -------
    ml_df : pd.DataFrame
        Clean dataframe containing labeled steps + engineered features.

    X : pd.DataFrame
        Features to use for ML.

    y : pd.Series
        Target labels (-1, 0, +1).

    groups : pd.Series
        trajectory_index, used for GroupShuffleSplit / GroupKFold.

    feature_groups : dict
        Lists of numerical, categorical, and text features.
    """

    df = df.copy()

    # ---------------------------------------------------------
    # 1. Basic cleanup
    # ---------------------------------------------------------

    df["content"] = df["content"].fillna("").astype(str)

    df["trajectory_index"] = pd.to_numeric(
        df["trajectory_index"],
        errors="coerce"
    )

    df["message_index"] = pd.to_numeric(
        df["message_index"],
        errors="coerce"
    )

    # ---------------------------------------------------------
    # 2. Extract tool name
    # ---------------------------------------------------------

    def extract_tool_name(tool_calls):
        if tool_calls is None:
            return None

        # Handles NaN safely
        if isinstance(tool_calls, float) and np.isnan(tool_calls):
            return None

        # Sometimes dataframe values may contain string representations
        if isinstance(tool_calls, str):
            try:
                tool_calls = ast.literal_eval(tool_calls)
            except (ValueError, SyntaxError):
                try:
                    tool_calls = json.loads(tool_calls)
                except Exception:
                    return None

        if isinstance(tool_calls, list) and len(tool_calls) > 0:
            first_call = tool_calls[0]

            if isinstance(first_call, dict):
                schema = first_call.get("schema", {})

                if isinstance(schema, dict):
                    return schema.get("name")

        return None

    if "tool_calls" in df.columns:
        df["tool_name"] = df["tool_calls"].apply(extract_tool_name)
    else:
        df["tool_name"] = None

    # ---------------------------------------------------------
    # 3. Tool-related features
    # ---------------------------------------------------------

    df["is_tool_call"] = df["tool_name"].notna().astype(int)

    # Count previous tool calls in the same trajectory.
    #
    # IMPORTANT:
    # This uses only information that would have been available
    # BEFORE the current action.
    df["_tool_counter"] = df["is_tool_call"]

    df["previous_tool_calls"] = (
        df.groupby("trajectory_index")["_tool_counter"]
          .cumsum()
          - df["_tool_counter"]
    )

    df["previous_tool_calls"] = (
        df["previous_tool_calls"]
        .fillna(0)
        .clip(lower=0)
    )

    df.drop(columns="_tool_counter", inplace=True)

    # ---------------------------------------------------------
    # 4. Text / structural features
    # ---------------------------------------------------------

    df["char_length"] = df["content"].str.len()

    df["word_count"] = (
        df["content"]
        .str.split()
        .str.len()
        .fillna(0)
    )

    df["line_count"] = (
        df["content"]
        .str.count(r"\n")
        .add(1)
    )

    df["digit_count"] = (
        df["content"]
        .str.count(r"\d")
    )

    df["question_mark_count"] = (
        df["content"]
        .str.count(r"\?")
    )

    # Avoid giving extremely long messages disproportionate influence
    df["log_char_length"] = np.log1p(df["char_length"])

    # ---------------------------------------------------------
    # 5. Position inside trajectory
    # ---------------------------------------------------------

    max_message_index = (
        df.groupby("trajectory_index")["message_index"]
          .transform("max")
    )

    df["relative_position"] = (
        df["message_index"] /
        max_message_index.replace(0, np.nan)
    )

    df["relative_position"] = (
        df["relative_position"]
        .fillna(0)
        .clip(0, 1)
    )

    df["trajectory_stage"] = pd.cut(
        df["relative_position"],
        bins=[-0.001, 0.33, 0.66, 1.0],
        labels=["early", "middle", "late"],
        include_lowest=True
    )

    # ---------------------------------------------------------
    # 6. Number of previous messages
    # ---------------------------------------------------------

    df["previous_messages"] = (
        df.groupby("trajectory_index")
          .cumcount()
    )

    # ---------------------------------------------------------
    # 7. Keep ONLY labeled model decisions
    # ---------------------------------------------------------

    ml_df = (
        df[df["label"].notna()]
        .copy()
        .reset_index(drop=True)
    )

    ml_df["label"] = (
        pd.to_numeric(
            ml_df["label"],
            errors="coerce"
        )
        .astype(int)
    )

    # ---------------------------------------------------------
    # 8. Fill feature missing values
    # ---------------------------------------------------------

    ml_df["tool_name"] = ml_df["tool_name"].fillna("NO_TOOL")

    ml_df["trajectory_stage"] = (
        ml_df["trajectory_stage"]
        .astype("object")
        .fillna("unknown")
    )

    # ---------------------------------------------------------
    # 9. Define feature groups
    # ---------------------------------------------------------

    numeric_features = [
        "message_index",
        "relative_position",
        "previous_messages",
        "previous_tool_calls",
        "char_length",
        "log_char_length",
        "word_count",
        "line_count",
        "digit_count",
        "question_mark_count",
        "is_tool_call",
    ]

    categorical_features = [
        "tool_name",
        "trajectory_stage",
    ]

    # Keep raw text available.
    # Do NOT necessarily use it in experiment #1.
    text_features = [
        "content",
    ]

    feature_groups = {
        "numeric": numeric_features,
        "categorical": categorical_features,
        "text": text_features,
    }

    # ---------------------------------------------------------
    # 10. X / y / groups
    # ---------------------------------------------------------

    X = ml_df[
        numeric_features
        + categorical_features
        + text_features
    ].copy()

    y = ml_df["label"].copy()

    groups = ml_df["trajectory_index"].copy()

    # ---------------------------------------------------------
    # 11. Sanity checks
    # ---------------------------------------------------------

    assert len(X) == len(y) == len(groups)

    assert not y.isna().any()

    print(f"Modeling rows: {len(ml_df):,}")
    print(f"Trajectories: {groups.nunique():,}")

    print("\nLabel distribution:")
    print(
        y.value_counts()
        .sort_index()
        .to_frame("count")
        .assign(
            percentage=lambda d:
                d["count"] / len(y) * 100
        )
    )

    print("\nFeature groups:")
    for name, cols in feature_groups.items():
        print(f"{name}: {cols}")

    return ml_df, X, y, groups, feature_groups

In [11]:
ml_df, X, y, groups, feature_groups = engineer_features(df)

Modeling rows: 6,881
Trajectories: 250

Label distribution:
       count  percentage
label                   
-1      1859   27.016422
 0       287    4.170905
 1      4735   68.812673

Feature groups:
numeric: ['message_index', 'relative_position', 'previous_messages', 'previous_tool_calls', 'char_length', 'log_char_length', 'word_count', 'line_count', 'digit_count', 'question_mark_count', 'is_tool_call']
categorical: ['tool_name', 'trajectory_stage']
text: ['content']


In [12]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

assert set(groups_train).isdisjoint(set(groups_test))

print("Train samples:", len(X_train))
print("Test samples:", len(X_test))

print("Train trajectories:", groups_train.nunique())
print("Test trajectories:", groups_test.nunique())

print("\nTrain distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Train samples: 5514
Test samples: 1367
Train trajectories: 200
Test trajectories: 50

Train distribution:
label
 1    0.693326
-1    0.264236
 0    0.042437
Name: proportion, dtype: float64

Test distribution:
label
 1    0.667154
-1    0.294075
 0    0.038771
Name: proportion, dtype: float64


In [13]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

dummy = DummyClassifier(
    strategy="most_frequent"
)

dummy.fit(X_train, y_train)

y_pred = dummy.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[-1, 0, 1],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred,
        labels=[-1, 0, 1]
    )
)

Accuracy: 0.6671543525969276
Macro F1: 0.2667836770513383

Classification report:
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00       402
           0       0.00      0.00      0.00        53
           1       0.67      1.00      0.80       912

    accuracy                           0.67      1367
   macro avg       0.22      0.33      0.27      1367
weighted avg       0.45      0.67      0.53      1367


Confusion matrix:
[[  0   0 402]
 [  0   0  53]
 [  0   0 912]]


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_features = feature_groups["numeric"]
categorical_features = feature_groups["categorical"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)


logreg = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

In [15]:
logreg.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[-1, 0, 1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['message_index','relative_position','previous_messages',...,'tool_name', 'trajectory_stage','content']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'dr

In [16]:
y_pred = logreg.predict(X_test)

In [17]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        y_pred,
        average="macro"
    )
)

print("\nClassification report:")

print(
    classification_report(
        y_test,
        y_pred,
        labels=[-1, 0, 1],
        zero_division=0
    )
)

print("\nConfusion matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred,
        labels=[-1, 0, 1]
    )
)

Accuracy: 0.7907827359180688
Macro F1: 0.5611119402985074

Classification report:
              precision    recall  f1-score   support

          -1       0.79      0.53      0.63       402
           0       0.55      0.11      0.19        53
           1       0.79      0.95      0.86       912

    accuracy                           0.79      1367
   macro avg       0.71      0.53      0.56      1367
weighted avg       0.78      0.79      0.77      1367


Confusion matrix:
[[212   4 186]
 [  8   6  39]
 [ 48   1 863]]


## This experiment asks:
Can simple structural information about an agent trajectory predict whether the next agent step is correct?

Your model sees things like:
- message_index
- relative_position
- previous_messages
- previous_tool_calls
- char_length
- word_count
- line_count
- digit_count
- question_mark_count
- is_tool_call
- tool_name
- trajectory_stage

but not the actual semantic meaning of the agent's message.

In [18]:
balanced_logreg = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

balanced_logreg.fit(X_train, y_train)

balanced_pred = balanced_logreg.predict(X_test)

In [19]:
print(
    "Accuracy:",
    accuracy_score(y_test, balanced_pred)
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        balanced_pred,
        average="macro"
    )
)

print(
    classification_report(
        y_test,
        balanced_pred,
        labels=[-1, 0, 1],
        zero_division=0
    )
)

print(
    confusion_matrix(
        y_test,
        balanced_pred,
        labels=[-1, 0, 1]
    )
)

Accuracy: 0.6027798098024872
Macro F1: 0.4958528473585586
              precision    recall  f1-score   support

          -1       0.54      0.61      0.57       402
           0       0.13      0.66      0.22        53
           1       0.85      0.60      0.70       912

    accuracy                           0.60      1367
   macro avg       0.51      0.62      0.50      1367
weighted avg       0.73      0.60      0.64      1367

[[244  66  92]
 [ 13  35   5]
 [197 170 545]]


In [20]:
numeric_no_position = [
    col for col in feature_groups["numeric"]
    if col not in {
        "message_index",
        "relative_position",
        "previous_messages",
    }
]

categorical_no_position = [
    col for col in feature_groups["categorical"]
    if col != "trajectory_stage"
]

In [21]:
preprocessor_no_position = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_no_position
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_no_position
        )
    ]
)

In [22]:
logreg_no_position = Pipeline([
    (
        "preprocessor",
        preprocessor_no_position
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

logreg_no_position.fit(X_train, y_train)

pred_no_position = logreg_no_position.predict(X_test)

In [23]:
print(
    "Accuracy:",
    accuracy_score(y_test, pred_no_position)
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        pred_no_position,
        average="macro"
    )
)

print(
    classification_report(
        y_test,
        pred_no_position,
        labels=[-1, 0, 1],
        zero_division=0
    )
)

print(
    confusion_matrix(
        y_test,
        pred_no_position,
        labels=[-1, 0, 1]
    )
)

Accuracy: 0.7746891002194587
Macro F1: 0.5381050525530111
              precision    recall  f1-score   support

          -1       0.77      0.48      0.59       402
           0       0.83      0.09      0.17        53
           1       0.77      0.94      0.85       912

    accuracy                           0.77      1367
   macro avg       0.79      0.51      0.54      1367
weighted avg       0.78      0.77      0.75      1367

[[194   0 208]
 [  6   5  42]
 [ 51   1 860]]


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),

    ("categorical",
     OneHotEncoder(handle_unknown="ignore"),
     categorical_features),

    ("text",
     TfidfVectorizer(
         max_features=5000,
         ngram_range=(1, 2),
         min_df=2
     ),
     "content")
])

logreg = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])
logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        y_pred,
        average="macro"
    )
)

print(
    classification_report(
        y_test,
        y_pred,
        labels=[-1, 0, 1],
        zero_division=0
    )
)

print(
    confusion_matrix(
        y_test,
        y_pred,
        labels=[-1, 0, 1]
    )
)

Accuracy: 0.817117776152158
Macro F1: 0.5945794604839962
              precision    recall  f1-score   support

          -1       0.79      0.65      0.71       402
           0       0.67      0.11      0.19        53
           1       0.83      0.93      0.88       912

    accuracy                           0.82      1367
   macro avg       0.76      0.57      0.59      1367
weighted avg       0.81      0.82      0.80      1367

[[263   2 137]
 [  9   6  38]
 [ 63   1 848]]


In [29]:
text_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

text_model.fit(
    X_train["content"].fillna(""),
    y_train
)

text_pred = text_model.predict(
    X_test["content"].fillna("")
)

print(
    classification_report(
        y_test,
        text_pred,
        labels=[-1, 0, 1],
        zero_division=0
    )
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        text_pred,
        average="macro"
    )
)

              precision    recall  f1-score   support

          -1       0.67      0.22      0.34       402
           0       0.00      0.00      0.00        53
           1       0.71      0.96      0.81       912

    accuracy                           0.70      1367
   macro avg       0.46      0.39      0.38      1367
weighted avg       0.67      0.70      0.64      1367

Macro F1: 0.3825651769087523


In [125]:
import pandas as pd
import numpy as np

def step_to_text(step):
    role = step_role(step)

    if isinstance(step, StepToolCall):
        calls = []

        for call in step.tool_calls:
            name = call.schema.get("name", "UNKNOWN_TOOL")
            arguments = call.schema.get("arguments", "")

            calls.append(
                f"{name}({arguments})"
            )

        call_text = "\n".join(calls)

        content = step.content or ""

        return (
            f"[{role}]\n"
            f"{content}\n"
            f"{call_text}"
        ).strip()

    if isinstance(step, ToolResultDescription):
        return (
            f"[{role} name={step.name}]\n"
            f"{step.content or ''}"
        ).strip()

    return (
        f"[{role}]\n"
        f"{getattr(step, 'content', '') or ''}"
    ).strip()

def step_role(step):
    if isinstance(step, StepSystemPrompt):
        return "SYSTEM"

    if isinstance(step, StepUser):
        return "USER"

    if isinstance(step, StepAgent):
        return "ASSISTANT"

    if isinstance(step, StepToolCall):
        return "TOOL_CALL"

    if isinstance(step, ToolResultDescription):
        return "TOOL_RESULT"

    return "UNKNOWN"

def build_context_dataset(
    dataset_path,
    context_window=8,
):
    """
    Create one row per LABELED model decision.

    Each row contains:
      - current step
      - previous context
      - history-only structural features
      - target label

    context_window:
        Number of previous messages to include.
        None = entire previous trajectory.
    """

    trajectory_data = Trajectory(dataset_path)

    trajectories = {}

    # -----------------------------------------
    # Collect steps by trajectory
    # -----------------------------------------

    for step in trajectory_data:
        trajectory_index = getattr(
            step,
            "trajectory_index",
            None
        )

        message_index = getattr(
            step,
            "message_index",
            None
        )

        if trajectory_index is None:
            continue

        if message_index is None:
            continue

        trajectories.setdefault(
            trajectory_index,
            []
        ).append(step)

    rows = []

    # -----------------------------------------
    # Process each trajectory independently
    # -----------------------------------------

    for trajectory_index, steps in trajectories.items():

        steps = sorted(
            steps,
            key=lambda s: s.message_index
        )

        history = []

        previous_tool_calls = 0
        previous_tool_results = 0
        previous_user_messages = 0
        previous_assistant_messages = 0

        for step in steps:

            label = getattr(step, "label", None)

            # ---------------------------------
            # Context BEFORE current step
            # ---------------------------------

            if context_window is None:
                selected_history = history
            else:
                selected_history = history[-context_window:]

            context_text = "\n\n".join(
                item["text"]
                for item in selected_history
            )

            current_text = step_to_text(step)

            # ---------------------------------
            # Only labeled steps become targets
            # ---------------------------------

            if label is not None:

                rows.append({
                    "trajectory_index":
                        trajectory_index,

                    "message_index":
                        step.message_index,

                    "label":
                        int(label),

                    "reason":
                        getattr(step, "reason", None),

                    "current_text":
                        current_text,

                    "context_text":
                        context_text,

                    # Current step type
                    "current_role":
                        step_role(step),

                    "is_tool_call":
                        int(
                            isinstance(
                                step,
                                StepToolCall
                            )
                        ),

                    # History-only features
                    "previous_messages":
                        len(history),

                    "previous_tool_calls":
                        previous_tool_calls,

                    "previous_tool_results":
                        previous_tool_results,

                    "previous_user_messages":
                        previous_user_messages,

                    "previous_assistant_messages":
                        previous_assistant_messages,

                    # Context size
                    "context_char_length":
                        len(context_text),

                    "context_word_count":
                        len(context_text.split()),

                    # Current step size
                    "current_char_length":
                        len(current_text),

                    "current_word_count":
                        len(current_text.split()),
                })

            # ---------------------------------
            # AFTER prediction:
            # current step becomes history
            # ---------------------------------

            history.append({
                "message_index":
                    step.message_index,

                "role":
                    step_role(step),

                "text":
                    current_text,
            })

            if isinstance(step, StepToolCall):
                previous_tool_calls += len(
                    step.tool_calls
                )

            elif isinstance(
                step,
                ToolResultDescription
            ):
                previous_tool_results += 1

            elif isinstance(step, StepUser):
                previous_user_messages += 1

            elif isinstance(step, StepAgent):
                previous_assistant_messages += 1

    return pd.DataFrame(rows)

In [126]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


def prepare_three_context_datasets(window):
    context_a = build_context_dataset(
        dataset,
        context_window=window,
    )

    context_b = build_context_dataset(
        dataset_tau2,
        context_window=window,
    )

    context_c = build_context_dataset(
        dataset_bfcl,
        context_window=window,
    )

    # Unique trajectory groups across datasets
    context_a["group_id"] = (
        "a_" + context_a["trajectory_index"].astype(str)
    )

    context_b["group_id"] = (
        "b_" + context_b["trajectory_index"].astype(str)
    )

    context_c["group_id"] = (
        "c_" + context_c["trajectory_index"].astype(str)
    )

    context_df = pd.concat(
        [context_a, context_b, context_c],
        ignore_index=True,
    )

    return context_df

In [127]:
def prepare_context_features(context_df):
    df = context_df.copy()

    df["current_text"] = (
        df["current_text"]
        .fillna("")
        .astype(str)
    )

    df["context_text"] = (
        df["context_text"]
        .fillna("")
        .astype(str)
    )

    df["current_role"] = (
        df["current_role"]
        .fillna("UNKNOWN")
        .astype(str)
    )

    df["label"] = pd.to_numeric(
        df["label"],
        errors="coerce",
    )

    df = (
        df[df["label"].notna()]
        .copy()
        .reset_index(drop=True)
    )

    df["label"] = df["label"].astype(int)

    df["log_context_char_length"] = np.log1p(
        df["context_char_length"]
    )

    df["log_current_char_length"] = np.log1p(
        df["current_char_length"]
    )

    numeric_features = [
        "message_index",
        "previous_messages",
        "previous_tool_calls",
        "previous_tool_results",
        "previous_user_messages",
        "previous_assistant_messages",
        "context_char_length",
        "context_word_count",
        "log_context_char_length",
        "current_char_length",
        "current_word_count",
        "log_current_char_length",
        "is_tool_call",
    ]

    categorical_features = [
        "current_role",
    ]

    X = df[
        numeric_features
        + categorical_features
        + ["current_text", "context_text"]
    ].copy()

    y = df["label"].copy()

    groups = df["group_id"].copy()

    return (
        df,
        X,
        y,
        groups,
        numeric_features,
        categorical_features,
    )

In [128]:
def make_model(
    numeric_features,
    categorical_features,
):
    preprocessor = ColumnTransformer([
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),

        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features,
        ),

        (
            "current_text",
            TfidfVectorizer(
                max_features=5000,
                ngram_range=(1, 2),
                min_df=2,
            ),
            "current_text",
        ),

        (
            "context_text",
            TfidfVectorizer(
                max_features=10000,
                ngram_range=(1, 2),
                min_df=2,
            ),
            "context_text",
        ),
    ])

    return Pipeline([
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                random_state=42,
            ),
        ),
    ])

In [129]:
windows = [
    1,
    2,
    4,
    8,
    16,
    32,
    None,   # full history
]

results = []

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for window in windows:

    print("\n" + "=" * 60)
    print(f"CONTEXT WINDOW: {window}")
    print("=" * 60)

    context_df = prepare_three_context_datasets(
        window
    )

    (
        ml_df,
        X,
        y,
        groups,
        numeric_features,
        categorical_features,
    ) = prepare_context_features(
        context_df
    )

    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(
            X,
            y,
            groups=groups,
        ),
        start=1,
    ):

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        model = make_model(
            numeric_features,
            categorical_features,
        )

        model.fit(
            X_train,
            y_train,
        )

        pred = model.predict(
            X_val
        )

        macro_f1 = f1_score(
            y_val,
            pred,
            average="macro",
        )

        fold_scores.append(
            macro_f1
        )

        print(
            f"Fold {fold}: "
            f"Macro F1 = {macro_f1:.3f}"
        )

    mean_score = np.mean(
        fold_scores
    )

    std_score = np.std(
        fold_scores
    )

    print(
        f"Mean: {mean_score:.3f}"
    )

    print(
        f"Std:  {std_score:.3f}"
    )

    results.append({
        "context_window": (
            "all"
            if window is None
            else window
        ),
        "mean_macro_f1":
            mean_score,
        "std_macro_f1":
            std_score,
        "min_macro_f1":
            np.min(fold_scores),
        "max_macro_f1":
            np.max(fold_scores),
    })


CONTEXT WINDOW: 1
Fold 1: Macro F1 = 0.595
Fold 2: Macro F1 = 0.532
Fold 3: Macro F1 = 0.575
Fold 4: Macro F1 = 0.612
Fold 5: Macro F1 = 0.618
Mean: 0.586
Std:  0.031

CONTEXT WINDOW: 2
Fold 1: Macro F1 = 0.593
Fold 2: Macro F1 = 0.547
Fold 3: Macro F1 = 0.605
Fold 4: Macro F1 = 0.642
Fold 5: Macro F1 = 0.622
Mean: 0.602
Std:  0.032

CONTEXT WINDOW: 4
Fold 1: Macro F1 = 0.586
Fold 2: Macro F1 = 0.541
Fold 3: Macro F1 = 0.599
Fold 4: Macro F1 = 0.631
Fold 5: Macro F1 = 0.591
Mean: 0.590
Std:  0.029

CONTEXT WINDOW: 8
Fold 1: Macro F1 = 0.589
Fold 2: Macro F1 = 0.542
Fold 3: Macro F1 = 0.571
Fold 4: Macro F1 = 0.620
Fold 5: Macro F1 = 0.579
Mean: 0.580
Std:  0.025

CONTEXT WINDOW: 16
Fold 1: Macro F1 = 0.596
Fold 2: Macro F1 = 0.540
Fold 3: Macro F1 = 0.586
Fold 4: Macro F1 = 0.634
Fold 5: Macro F1 = 0.581
Mean: 0.587
Std:  0.030

CONTEXT WINDOW: 32
Fold 1: Macro F1 = 0.589
Fold 2: Macro F1 = 0.537
Fold 3: Macro F1 = 0.589
Fold 4: Macro F1 = 0.634
Fold 5: Macro F1 = 0.570
Mean: 0.584
St

In [130]:
results_df = pd.DataFrame(
    results
)

results_df.sort_values(
    "mean_macro_f1",
    ascending=False,
)
results_df

,context_window,mean_macro_f1,std_macro_f1,min_macro_f1,max_macro_f1
0,1,0.586257,0.031063,0.531665,0.617541
1,2,0.601951,0.032161,0.546852,0.642432
2,4,0.589608,0.028976,0.540733,0.630823
3,8,0.580073,0.025141,0.542036,0.619553
4,16,0.587447,0.030349,0.539585,0.634229
5,32,0.583677,0.031357,0.537099,0.633593
6,all,0.584321,0.031584,0.539635,0.636958


In [131]:
# 1     → .586
# 2     → .602  ← best
# 4     → .590
# 8     → .580
# 16    → .587
# 32    → .584
# all   → .584

CONTEXT_WINDOW = 2

context_a = build_context_dataset(
    dataset,
    context_window=CONTEXT_WINDOW,
)

context_b = build_context_dataset(
    dataset_tau2,
    context_window=CONTEXT_WINDOW,
)

context_c = build_context_dataset(
    dataset_bfcl,
    context_window=CONTEXT_WINDOW,
)

# Make trajectory IDs unique across the three datasets
context_a["group_id"] = (
    "a_" + context_a["trajectory_index"].astype(str)
)

context_b["group_id"] = (
    "b_" + context_b["trajectory_index"].astype(str)
)

context_c["group_id"] = (
    "c_" + context_c["trajectory_index"].astype(str)
)

context_df = pd.concat(
    [
        context_a,
        context_b,
        context_c,
    ],
    ignore_index=True,
)

print("Rows:", len(context_df))
print("Trajectories:", context_df["group_id"].nunique())
print(context_df["label"].value_counts())

Rows: 6881
Trajectories: 750
label
 1    4735
-1    1859
 0     287
Name: count, dtype: int64


In [132]:
def prepare_ablation_data(df):
    df = df.copy()

    # -----------------------------------------
    # Basic cleanup
    # -----------------------------------------

    df["current_text"] = (
        df["current_text"]
        .fillna("")
        .astype(str)
    )

    df["context_text"] = (
        df["context_text"]
        .fillna("")
        .astype(str)
    )

    df["current_role"] = (
        df["current_role"]
        .fillna("UNKNOWN")
        .astype(str)
    )

    df["label"] = pd.to_numeric(
        df["label"],
        errors="coerce",
    )

    df = (
        df[df["label"].notna()]
        .copy()
        .reset_index(drop=True)
    )

    df["label"] = df["label"].astype(int)

    # -----------------------------------------
    # Extra structural features
    # -----------------------------------------

    df["log_context_char_length"] = np.log1p(
        df["context_char_length"]
    )

    df["log_current_char_length"] = np.log1p(
        df["current_char_length"]
    )

    # -----------------------------------------
    # Structural feature groups
    # -----------------------------------------

    numeric_features = [
        "message_index",
        "previous_messages",
        "previous_tool_calls",
        "previous_tool_results",
        "previous_user_messages",
        "previous_assistant_messages",
        "context_char_length",
        "context_word_count",
        "log_context_char_length",
        "current_char_length",
        "current_word_count",
        "log_current_char_length",
        "is_tool_call",
    ]

    categorical_features = [
        "current_role",
    ]

    X = df[
        numeric_features
        + categorical_features
        + [
            "current_text",
            "context_text",
        ]
    ].copy()

    y = df["label"].copy()

    groups = df["group_id"].copy()

    return (
        df,
        X,
        y,
        groups,
        numeric_features,
        categorical_features,
    )

In [133]:
(
    ml_df,
    X,
    y,
    groups,
    numeric_features,
    categorical_features,
) = prepare_ablation_data(context_df)

In [134]:
experiments = {
    "structural_only": {
        "structural": True,
        "current_text": False,
        "context_text": False,
    },

    "current_text_only": {
        "structural": False,
        "current_text": True,
        "context_text": False,
    },

    "context_text_only": {
        "structural": False,
        "current_text": False,
        "context_text": True,
    },

    "structural_current": {
        "structural": True,
        "current_text": True,
        "context_text": False,
    },

    "structural_context": {
        "structural": True,
        "current_text": False,
        "context_text": True,
    },

    "current_context": {
        "structural": False,
        "current_text": True,
        "context_text": True,
    },

    "all_features": {
        "structural": True,
        "current_text": True,
        "context_text": True,
    },
}

In [135]:
def make_ablation_model(
    config,
    numeric_features,
    categorical_features,
):
    transformers = []

    # -----------------------------------------
    # Structural features
    # -----------------------------------------

    if config["structural"]:

        transformers.append(
            (
                "numeric",
                StandardScaler(),
                numeric_features,
            )
        )

        transformers.append(
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical_features,
            )
        )

    # -----------------------------------------
    # Current step text
    # -----------------------------------------

    if config["current_text"]:

        transformers.append(
            (
                "current_text",
                TfidfVectorizer(
                    max_features=5000,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "current_text",
            )
        )

    # -----------------------------------------
    # Previous context
    # -----------------------------------------

    if config["context_text"]:

        transformers.append(
            (
                "context_text",
                TfidfVectorizer(
                    max_features=10000,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
                "context_text",
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers
    )

    model = Pipeline([
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=3000,
                random_state=42,
            ),
        ),
    ])

    return model

In [136]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

cv_splits = list(
    cv.split(
        X,
        y,
        groups=groups,
    )
)

In [104]:
results = []

for experiment_name, config in experiments.items():

    print("\n" + "=" * 70)
    print(experiment_name)
    print("=" * 70)

    fold_scores = []
    fold_error_f1 = []
    fold_error_recall = []

    for fold, (train_idx, val_idx) in enumerate(
        cv_splits,
        start=1,
    ):

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        model = make_ablation_model(
            config=config,
            numeric_features=numeric_features,
            categorical_features=categorical_features,
        )

        model.fit(
            X_train,
            y_train,
        )

        pred = model.predict(
            X_val
        )

        # Main metric
        macro_f1 = f1_score(
            y_val,
            pred,
            average="macro",
        )

        # Specifically measure error class (-1)
        error_f1 = f1_score(
            y_val,
            pred,
            labels=[-1],
            average="macro",
            zero_division=0,
        )

        # Recall specifically for -1
        actual_errors = (y_val == -1)
        predicted_errors = (pred == -1)

        true_positive_errors = (
            actual_errors
            & predicted_errors
        ).sum()

        error_recall = (
            true_positive_errors
            / actual_errors.sum()
            if actual_errors.sum() > 0
            else 0
        )

        fold_scores.append(
            macro_f1
        )

        fold_error_f1.append(
            error_f1
        )

        fold_error_recall.append(
            error_recall
        )

        print(
            f"Fold {fold}: "
            f"Macro F1={macro_f1:.3f} | "
            f"-1 F1={error_f1:.3f} | "
            f"-1 Recall={error_recall:.3f}"
        )

    results.append({
        "experiment":
            experiment_name,

        "mean_macro_f1":
            np.mean(fold_scores),

        "std_macro_f1":
            np.std(fold_scores),

        "mean_error_f1":
            np.mean(fold_error_f1),

        "mean_error_recall":
            np.mean(fold_error_recall),

        "min_macro_f1":
            np.min(fold_scores),

        "max_macro_f1":
            np.max(fold_scores),
    })


structural_only
Fold 1: Macro F1=0.454 | -1 F1=0.512 | -1 Recall=0.385
Fold 2: Macro F1=0.430 | -1 F1=0.447 | -1 Recall=0.329
Fold 3: Macro F1=0.377 | -1 F1=0.303 | -1 Recall=0.201
Fold 4: Macro F1=0.394 | -1 F1=0.347 | -1 Recall=0.223
Fold 5: Macro F1=0.400 | -1 F1=0.361 | -1 Recall=0.250

current_text_only
Fold 1: Macro F1=0.574 | -1 F1=0.624 | -1 Recall=0.526
Fold 2: Macro F1=0.468 | -1 F1=0.556 | -1 Recall=0.493
Fold 3: Macro F1=0.528 | -1 F1=0.586 | -1 Recall=0.488
Fold 4: Macro F1=0.584 | -1 F1=0.581 | -1 Recall=0.476
Fold 5: Macro F1=0.550 | -1 F1=0.617 | -1 Recall=0.524

context_text_only
Fold 1: Macro F1=0.478 | -1 F1=0.552 | -1 Recall=0.458
Fold 2: Macro F1=0.466 | -1 F1=0.544 | -1 Recall=0.469
Fold 3: Macro F1=0.464 | -1 F1=0.535 | -1 Recall=0.437
Fold 4: Macro F1=0.489 | -1 F1=0.573 | -1 Recall=0.473
Fold 5: Macro F1=0.490 | -1 F1=0.603 | -1 Recall=0.522

structural_current
Fold 1: Macro F1=0.600 | -1 F1=0.692 | -1 Recall=0.612
Fold 2: Macro F1=0.514 | -1 F1=0.644 | -1 Rec

In [137]:
ablation_results = pd.DataFrame(
    results
)

ablation_results = (
    ablation_results
    .sort_values(
        "mean_macro_f1",
        ascending=False,
    )
    .reset_index(drop=True)
)

ablation_results

,context_window,mean_macro_f1,std_macro_f1,min_macro_f1,max_macro_f1
0,2,0.601951,0.032161,0.546852,0.642432
1,4,0.589608,0.028976,0.540733,0.630823
2,16,0.587447,0.030349,0.539585,0.634229
3,1,0.586257,0.031063,0.531665,0.617541
4,all,0.584321,0.031584,0.539635,0.636958
5,32,0.583677,0.031357,0.537099,0.633593
6,8,0.580073,0.025141,0.542036,0.619553


In [138]:
# context window = 2
# current text
# + previous context
# + structural features
# + Logistic Regression

best_config = {
    "structural": True,
    "current_text": True,
    "context_text": True,
}

In [139]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

# Store one prediction per row
oof_pred = np.full(
    len(y),
    fill_value=np.nan,
)

# Optional: store fold number
oof_fold = np.full(
    len(y),
    fill_value=-1,
    dtype=int,
)

for fold, (train_idx, val_idx) in enumerate(
    cv.split(
        X,
        y,
        groups=groups,
    ),
    start=1,
):

    print(f"Running fold {fold}...")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = make_ablation_model(
        config=best_config,
        numeric_features=numeric_features,
        categorical_features=categorical_features,
    )

    model.fit(
        X_train,
        y_train,
    )

    pred = model.predict(
        X_val
    )

    # Save predictions back into original row positions
    oof_pred[val_idx] = pred
    oof_fold[val_idx] = fold

Running fold 1...
Running fold 2...
Running fold 3...
Running fold 4...
Running fold 5...


In [140]:
assert not np.isnan(oof_pred).any()
oof_pred = oof_pred.astype(int)

In [141]:
error_df = ml_df.copy()

error_df["actual_label"] = y.to_numpy()
error_df["predicted_label"] = oof_pred
error_df["cv_fold"] = oof_fold

error_df["correct_prediction"] = (
    error_df["actual_label"]
    == error_df["predicted_label"]
)

print(
    "OOF Macro F1:",
    f1_score(
        error_df["actual_label"],
        error_df["predicted_label"],
        average="macro",
    )
)

print(
    classification_report(
        error_df["actual_label"],
        error_df["predicted_label"],
        labels=[-1, 0, 1],
        zero_division=0,
    )
)

print(
    confusion_matrix(
        error_df["actual_label"],
        error_df["predicted_label"],
        labels=[-1, 0, 1],
    )
)

OOF Macro F1: 0.6026138641009203
              precision    recall  f1-score   support

          -1       0.78      0.63      0.69      1859
           0       0.52      0.14      0.22       287
           1       0.84      0.94      0.89      4735

    accuracy                           0.82      6881
   macro avg       0.71      0.57      0.60      6881
weighted avg       0.81      0.82      0.81      6881

[[1164   28  667]
 [  70   41  176]
 [ 259   10 4466]]


In [142]:
false_negatives = error_df[
    (error_df["actual_label"] == -1)
    & (error_df["predicted_label"] == 1)
].copy()
print(
    "False negatives:",
    len(false_negatives)
)

False negatives: 667


In [143]:
false_negatives[
    [
        "trajectory_index",
        "message_index",
        "current_role",
        "context_text",
        "current_text",
        "actual_label",
        "predicted_label",
    ]
].head(20)

,trajectory_index,message_index,current_role,context_text,current_text,actual_label,predicted_label
3,0,8,ASSISTANT,"[TOOL_CALL]\n\nsearch({""query_list"": [""Melbour...",[ASSISTANT]\nThe Australian city founded in 18...,-1,1
6,1,6,ASSISTANT,"[TOOL_CALL]\n\nsearch({""query"": ""Australian ci...",[ASSISTANT]\nThe Australian city founded in 18...,-1,1
11,2,10,ASSISTANT,[TOOL_CALL]\n搜索结果没有直接给出答案。我们来搜索一下“boarding sch...,[ASSISTANT]\n<answer> Adelaide </answer>,-1,1
15,4,2,ASSISTANT,,[ASSISTANT]\n<think>We need an Australian city...,-1,1
29,9,2,ASSISTANT,,[ASSISTANT]\n<think>We need to determine wheth...,-1,1
70,29,2,ASSISTANT,,[ASSISTANT]\nSorry — I can’t share internal ch...,-1,1
83,35,8,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""Cressid...","[TOOL_CALL]\n\nsearch({""query_list"": [""Cressid...",-1,1
84,35,10,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""Cressid...","[TOOL_CALL]\n\nsearch({""query_list"": [""Cressid...",-1,1
85,35,12,ASSISTANT,"[TOOL_CALL]\n\nsearch({""query_list"": [""Cressid...","[ASSISTANT]\nAfter multiple searches, the info...",-1,1
88,36,6,ASSISTANT,[TOOL_CALL]\n<Cressida Bonas> is the youngest ...,[ASSISTANT]\nAfter conducting multiple searche...,-1,1


In [144]:
for i, (_, row) in enumerate(
    false_negatives.head(20).iterrows(),
    start=1,
):

    print("\n" + "=" * 100)
    print(f"FALSE NEGATIVE #{i}")
    print("=" * 100)

    print(
        f"Trajectory: {row['trajectory_index']} | "
        f"Message: {row['message_index']} | "
        f"Role: {row['current_role']}"
    )

    print("\n--- PREVIOUS CONTEXT ---")
    print(
        row["context_text"]
    )

    print("\n--- CURRENT STEP ---")
    print(
        row["current_text"]
    )

    print("\nActual: -1")
    print("Predicted: +1")


FALSE NEGATIVE #1
Trajectory: 0 | Message: 8 | Role: ASSISTANT

--- PREVIOUS CONTEXT ---
[TOOL_CALL]

search({"query_list": ["Melbourne boarding school founded by Australian Prime Minister named after London school"]})

[TOOL_RESULT name=search]
{"result": [[{"document": {"id": "6327952", "contents": "\"Melbourne Grammar School\"\nMelbourne Grammar School Melbourne Grammar School is an independent, Anglican, day and boarding school predominantly for boys, located in South Yarra and Caulfield, suburbs of Melbourne, Victoria, Australia. Founded on 7 April 1858 as the Melbourne Church of England Grammar School, the school has a non-selective enrolment policy and currently caters for approximately 1,800 students from Prep to Year 12, including 120 boarders from Years 7 to 12. Melbourne Grammar is affiliated with the Headmasters' and Headmistresses' Conference, the Association of Heads of Independent Schools of Australia (AHISA), the Independent Primary School Heads of Australia (IPSHA), t

In [145]:
false_positives = error_df[
    (error_df["actual_label"] == 1)
    & (error_df["predicted_label"] == -1)
].copy()

zero_errors = error_df[
    (error_df["actual_label"] == 0)
    & (error_df["predicted_label"] != 0)
].copy()

error_summary = pd.DataFrame({
    "error_type": [
        "-1 predicted as +1",
        "+1 predicted as -1",
        "0 predicted incorrectly",
    ],
    "count": [
        len(false_negatives),
        len(false_positives),
        len(zero_errors),
    ],
})

error_summary

,error_type,count
0,-1 predicted as +1,667
1,+1 predicted as -1,259
2,0 predicted incorrectly,246


In [157]:
print(ml_df.columns.tolist())
columns = [
    "trajectory_index",
    "message_index",
    "current_role",
    "context_text",
    "current_text",
    "reason",
    "actual_label",
    "predicted_label",
]

missed_failures = error_df[
    (error_df["actual_label"] == -1)
    & (error_df["predicted_label"] == 1)
].copy()

sample_missed = missed_failures.sample(
    n=min(100, len(missed_failures)),
    random_state=42
)

sample_missed["error_category"] = pd.NA

review = sample_missed[
    [
        "trajectory_index",
        "message_index",
        "current_role",
        "context_text",
        "current_text",
        "reason",
        "actual_label",
        "predicted_label",
    ]
].copy()

review["error_category"] = pd.NA

review.reset_index(drop=True, inplace=True)
sample_missed

['trajectory_index', 'message_index', 'label', 'reason', 'current_text', 'context_text', 'current_role', 'is_tool_call', 'previous_messages', 'previous_tool_calls', 'previous_tool_results', 'previous_user_messages', 'previous_assistant_messages', 'context_char_length', 'context_word_count', 'current_char_length', 'current_word_count', 'group_id', 'log_context_char_length', 'log_current_char_length']


,trajectory_index,message_index,label,reason,current_text,context_text,current_role,is_tool_call,previous_messages,previous_tool_calls,...,current_char_length,current_word_count,group_id,log_context_char_length,log_current_char_length,actual_label,predicted_label,cv_fold,correct_prediction,error_category
6775,240,21,-1,Annotated -1: Repeats the same invoice retriev...,"[TOOL_CALL]\n\nretrieve_invoice({""access_token...","[TOOL_CALL]\n\nretrieve_invoice({""access_token...",TOOL_CALL,1,14,6,...,60,3,c_240,5.099866,4.110874,-1,1,1,False,<NA>
1166,40,13,-1,Annotated -1: Attempts an update that reduces ...,[TOOL_CALL]\n\nupdate_reservation_passengers({...,[TOOL_RESULT name=calculate]\n633.33\n\n[ASSIS...,TOOL_CALL,1,8,2,...,226,16,b_40,6.139885,5.424950,-1,1,3,False,<NA>
6504,208,9,-1,Annotated -1: Reports successful booking/invoi...,[ASSISTANT]\nA **business class** flight from ...,"[TOOL_CALL]\n\nretrieve_invoice({""access_token...",ASSISTANT,0,6,3,...,515,70,c_208,5.789960,6.246107,-1,1,1,False,<NA>
2008,109,3,-1,Annotated -1: Invents ZIP code (00000) not pro...,"[TOOL_CALL]\n\nfind_user_id_by_name_zip({""firs...",[ASSISTANT]\nHi! How can I help you today?,TOOL_CALL,1,1,0,...,103,7,b_109,3.737670,4.644391,-1,1,3,False,<NA>
6509,209,9,-1,Annotated -1: Reports successful booking/invoi...,[ASSISTANT]\nThe business class flight from Sa...,"[TOOL_CALL]\n\nretrieve_invoice({""access_token...",ASSISTANT,0,6,3,...,423,64,c_209,5.730100,6.049733,-1,1,3,False,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4023,239,5,-1,Annotated -1: the assistant responds 'Thanks —...,[ASSISTANT]\nThanks — I found your account (Jo...,"[TOOL_CALL]\n\nget_customer_by_phone({""phone_n...",ASSISTANT,0,3,1,...,382,60,b_239,6.548219,5.948035,-1,1,3,False,<NA>
973,23,15,-1,Annotated -1: Confirms completion while omitti...,[ASSISTANT]\nYour reservation YAX4DR has been ...,"[TOOL_CALL]\n\nupdate_reservation_baggages({""n...",ASSISTANT,0,11,4,...,333,54,b_23,6.869014,5.811141,-1,1,3,False,<NA>
5026,59,34,-1,Annotated -1: Posted extra hashtags not reques...,"[TOOL_CALL]\n\npost_tweet({""content"": ""Excited...",[TOOL_CALL]\n\nposting_get_login_status({})\n\...,TOOL_CALL,1,25,11,...,114,11,c_59,4.700480,4.744932,-1,1,1,False,<NA>
2468,144,15,-1,"Annotated -1: this step responds '{""message"":""...","[ASSISTANT]\n{""message"":""I tried to look up or...","[TOOL_CALL]\n\nget_order_details({""order_id"": ...",ASSISTANT,0,10,3,...,346,63,b_144,4.762174,5.849325,-1,1,2,False,<NA>


In [ ]:
def initialize_error_annotations(df):
    df = df.copy()

    df["error_category"] = df["reason"].copy()
    df["annotation_source"] = "original_reason"

    return df

# TODO: Annotate with codex
sample_missed = initialize_error_annotations(sample_missed)

In [161]:
category_summary = (
    sample_missed["error_category"]
    .value_counts(dropna=False)
    .rename_axis("error_category")
    .reset_index(name="count")
)

category_summary["percentage"] = (
    category_summary["count"]
    / category_summary["count"].sum()
    * 100
)

category_summary

,error_category,count,percentage
0,Annotated -1: Repeats the same invoice retriev...,1,1.0
1,Annotated -1: Attempts an update that reduces ...,1,1.0
2,Annotated -1: Reports successful booking/invoi...,1,1.0
3,Annotated -1: Invents ZIP code (00000) not pro...,1,1.0
4,Annotated -1: Reports successful booking/invoi...,1,1.0
...,...,...,...
95,Annotated -1: the assistant responds 'Thanks —...,1,1.0
96,Annotated -1: Confirms completion while omitti...,1,1.0
97,Annotated -1: Posted extra hashtags not reques...,1,1.0
98,"Annotated -1: this step responds '{""message"":""...",1,1.0


In [162]:
context_a = context_a.copy()
context_b = context_b.copy()
context_c = context_c.copy()

context_a["group_id"] = "a_" + context_a["trajectory_index"].astype(str)
context_b["group_id"] = "b_" + context_b["trajectory_index"].astype(str)
context_c["group_id"] = "c_" + context_c["trajectory_index"].astype(str)

In [163]:
def prepare_cross_dataset(df):
    df = df.copy()

    df["current_text"] = (
        df["current_text"]
        .fillna("")
        .astype(str)
    )

    df["context_text"] = (
        df["context_text"]
        .fillna("")
        .astype(str)
    )

    df["current_role"] = (
        df["current_role"]
        .fillna("UNKNOWN")
        .astype(str)
    )

    df["label"] = pd.to_numeric(
        df["label"],
        errors="coerce"
    )

    df = (
        df[df["label"].notna()]
        .copy()
        .reset_index(drop=True)
    )

    df["label"] = df["label"].astype(int)

    df["log_context_char_length"] = np.log1p(
        df["context_char_length"]
    )

    df["log_current_char_length"] = np.log1p(
        df["current_char_length"]
    )

    numeric_features = [
        "message_index",
        "previous_messages",
        "previous_tool_calls",
        "previous_tool_results",
        "previous_user_messages",
        "previous_assistant_messages",
        "context_char_length",
        "context_word_count",
        "log_context_char_length",
        "current_char_length",
        "current_word_count",
        "log_current_char_length",
        "is_tool_call",
    ]

    categorical_features = [
        "current_role",
    ]

    X = df[
        numeric_features
        + categorical_features
        + [
            "current_text",
            "context_text",
        ]
    ].copy()

    y = df["label"].copy()

    return (
        df,
        X,
        y,
        numeric_features,
        categorical_features,
    )

In [164]:
def run_cross_dataset_experiment(
    train_dfs,
    test_df,
    experiment_name,
):
    train_df = pd.concat(
        train_dfs,
        ignore_index=True,
    )

    (
        train_ml,
        X_train,
        y_train,
        numeric_features,
        categorical_features,
    ) = prepare_cross_dataset(train_df)

    (
        test_ml,
        X_test,
        y_test,
        _,
        _,
    ) = prepare_cross_dataset(test_df)

    model = make_ablation_model(
        config={
            "structural": True,
            "current_text": True,
            "context_text": True,
        },
        numeric_features=numeric_features,
        categorical_features=categorical_features,
    )

    model.fit(
        X_train,
        y_train,
    )

    pred = model.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test,
        pred,
    )

    macro_f1 = f1_score(
        y_test,
        pred,
        average="macro",
    )

    error_f1 = f1_score(
        y_test,
        pred,
        labels=[-1],
        average="macro",
        zero_division=0,
    )

    error_recall = (
        ((y_test == -1) & (pred == -1)).sum()
        / (y_test == -1).sum()
    )

    print("\n" + "=" * 80)
    print(experiment_name)
    print("=" * 80)

    print("Train rows:", len(train_ml))
    print("Test rows:", len(test_ml))

    print("\nTrain distribution:")
    print(
        y_train.value_counts(normalize=True)
        .sort_index()
    )

    print("\nTest distribution:")
    print(
        y_test.value_counts(normalize=True)
        .sort_index()
    )

    print("\nAccuracy:", accuracy)
    print("Macro F1:", macro_f1)
    print("-1 F1:", error_f1)
    print("-1 Recall:", error_recall)

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            pred,
            labels=[-1, 0, 1],
            zero_division=0,
        )
    )

    print("Confusion matrix:")
    print(
        confusion_matrix(
            y_test,
            pred,
            labels=[-1, 0, 1],
        )
    )

    return {
        "experiment": experiment_name,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "error_f1": error_f1,
        "error_recall": error_recall,
        "train_rows": len(train_ml),
        "test_rows": len(test_ml),
    }

In [165]:
results = []

results.append(
    run_cross_dataset_experiment(
        train_dfs=[context_a, context_b],
        test_df=context_c,
        experiment_name="A+B -> C",
    )
)

results.append(
    run_cross_dataset_experiment(
        train_dfs=[context_a, context_c],
        test_df=context_b,
        experiment_name="A+C -> B",
    )
)

results.append(
    run_cross_dataset_experiment(
        train_dfs=[context_b, context_c],
        test_df=context_a,
        experiment_name="B+C -> A",
    )
)


A+B -> C
Train rows: 4291
Test rows: 2590

Train distribution:
label
-1    0.300396
 0    0.042647
 1    0.656956
Name: proportion, dtype: float64

Test distribution:
label
-1    0.220077
 0    0.040154
 1    0.739768
Name: proportion, dtype: float64

Accuracy: 0.5007722007722007
Macro F1: 0.30480582726555144
-1 F1: 0.28229937393284005
-1 Recall: 0.43508771929824563

Classification report:
              precision    recall  f1-score   support

          -1       0.21      0.44      0.28       570
           0       0.00      0.00      0.00       104
           1       0.75      0.55      0.63      1916

    accuracy                           0.50      2590
   macro avg       0.32      0.33      0.30      2590
weighted avg       0.60      0.50      0.53      2590

Confusion matrix:
[[ 248    0  322]
 [  72    0   32]
 [ 867    0 1049]]

A+C -> B
Train rows: 3324
Test rows: 3557

Train distribution:
label
-1    0.225331
 0    0.048135
 1    0.726534
Name: proportion, dtype: float64

Tes

In [166]:
cross_dataset_results = pd.DataFrame(
    results
)

cross_dataset_results

,experiment,accuracy,macro_f1,error_f1,error_recall,train_rows,test_rows
0,A+B -> C,0.500772,0.304806,0.282299,0.435088,4291,2590
1,A+C -> B,0.633399,0.407269,0.496940,0.548649,3324,3557
2,B+C -> A,0.666213,0.411466,0.458221,0.474860,6147,734


Yes. At this point you have enough experiments to see the **full picture**. There are really three separate questions: **what data setup is correct, what features/context work, and how good the model is/generalizes.**

### Current picture

| Component                 | What you tested                                   | Current conclusion                              |
| ------------------------- | ------------------------------------------------- | ----------------------------------------------- |
| **Dataset**               | 3 datasets, ~750 trajectories                     | Good diversity for a baseline                   |
| **Target**                | `-1 / 0 / +1`                                     | Valid, but `0` is very rare (~4%) and difficult |
| **Split strategy**        | trajectory-grouped split/CV                       | **Correct and essential**                       |
| **Context**               | 1, 2, 4, 8, 16, 32, all                           | **2 previous steps is currently best**          |
| **Structural features**   | position, role, counts, lengths, tool information | Helpful, but weak alone                         |
| **Current text**          | TF-IDF                                            | Strong individual signal                        |
| **Previous context text** | TF-IDF                                            | Useful complementary signal                     |
| **Best features**         | structural + current + context                    | **Best configuration**                          |
| **Classifier**            | Logistic Regression                               | Good interpretable baseline                     |
| **Mixed grouped CV**      | 5-fold                                            | **Macro F1 ≈ 0.602 ± 0.032**                    |
| **Error detection**       | class `-1`                                        | F1 ≈ **0.694**, recall ≈ **0.626**              |
| **Cross-dataset**         | train 2 → test 1                                  | **Major weakness**                              |

## 1. Your dataset design is good now

The important unit is the **trajectory**, not an individual row.

You correctly fixed the collision:

```text
A: a_0 ... a_249
B: b_0 ... b_249
C: c_0 ... c_249
```

That gives approximately:

```text
750 unique trajectories
6,881 labeled steps
```

and your class distribution is approximately:

```text
+1   68.8%
-1   27.0%
 0    4.2%
```

The imbalance is manageable for `-1`, but `0` is genuinely difficult because there are very few examples.

Most importantly, your validation strategy is now correct:

```text
BAD:

same trajectory
 ├─ some steps → train
 └─ other steps → test


GOOD:

trajectory 123 → entirely train
trajectory 124 → entirely validation
trajectory 125 → entirely train
```

Keep this rule for **every future model**.

---

## 2. Context genuinely helps

Your context-window experiment was:

| Previous steps |  Macro F1 |
| -------------: | --------: |
|              1 |     0.586 |
|          **2** | **0.602** |
|              4 |     0.590 |
|              8 |     0.580 |
|             16 |     0.587 |
|             32 |     0.584 |
|            All |     0.584 |

This is a useful result.

More history is **not automatically better**.

Your current conclusion should be:

> Short, local conversational history is most useful for step-level error detection, with a two-step context window achieving the highest mean Macro F1.

Don't claim that `2` is universally optimal. The differences are small enough that the exact optimum could change with more data or another model.

For now, however, **freeze `context_window=2`**.

---

## 3. Your feature ablation is particularly good

This is one of your strongest experiments:

| Features                           |  Macro F1 |
| ---------------------------------- | --------: |
| Structural only                    |     0.411 |
| Context only                       |     0.477 |
| Current text only                  |     0.541 |
| Structural + context               |     0.516 |
| Structural + current               |     0.563 |
| Current + context                  |     0.588 |
| **Structural + current + context** | **0.602** |

This tells a coherent story.

### Structural information alone isn't enough

```text
0.411
```

Things like:

```text
message position
number of previous calls
message length
role
is_tool_call
```

contain signal, but they can't determine correctness by themselves.

### Current action is important

```text
current text only = 0.541
```

The model can often identify suspicious/correct actions from the current action itself.

### Context adds additional information

This comparison is especially important:

```text
structural + current
0.563

        ↓ + context

all
0.602
```

That's roughly:

```text
+0.039 Macro F1
```

And:

```text
current only
0.541

current + context
0.588
```

So context isn't just making the input longer. It provides useful predictive information.

### Your best representation currently is therefore

```text
                 ┌─ structural features
                 │
trajectory ──────┼─ previous 2 steps
                 │
                 └─ current step
                         ↓
                       TF-IDF
                         +
                   numeric/categorical
                         ↓
                Logistic Regression
```

That's a perfectly reasonable classical ML baseline.

---

## 4. Your model is good **as a baseline**

Your grouped CV result:

```text
Macro F1 = 0.602 ± 0.032
```

is substantially better than your dummy baseline:

```text
Macro F1 ≈ 0.267
```

So the model is definitely learning something.

And your error class:

```text
-1 F1     ≈ 0.694
-1 recall ≈ 0.626
```

is encouraging.

But it's not yet a highly reliable error detector because approximately **37% of erroneous steps are still missed**.

Class `0` is also a significant problem because it is both rare and difficult.

---

# 5. The biggest weakness is NOT mixed-CV performance

This is now the most important discovery you've made.

Mixed grouped CV:

```text
Macro F1 ≈ 0.602
```

But leave-one-dataset-out gives:

| Train | Unseen test |  Macro F1 | Error F1 | Error recall |
| ----- | ----------- | --------: | -------: | -----------: |
| A + B | C           | **0.305** |    0.282 |        0.435 |
| A + C | B           | **0.407** |    0.497 |        0.549 |
| B + C | A           | **0.411** |    0.458 |        0.475 |

That's a big finding.

Your model can generalize to **new trajectories drawn from known benchmark distributions** reasonably well:

```text
Mixed grouped CV ≈ 0.60
```

but struggles when confronted with an **entirely unseen benchmark/domain**:

```text
Cross-dataset ≈ 0.30–0.41
```

So there are actually two notions of generalization here:

```text
                    Generalization
                         │
             ┌───────────┴───────────┐
             │                       │
      New trajectories         New benchmark
      from known domains       / environment
             │                       │
          ~0.60                  ~0.30–0.41
          GOODER                   WEAK
```

That's probably more interesting than simply obtaining a higher mixed-CV score.

---

# 6. Why is cross-dataset performance weak?

Your ablation gives us a strong clue.

Most of the predictive performance comes from **text**.

TF-IDF represents lexical patterns. Imagine A/B contains:

```text
cancel_reservation
update_reservation
flight
passenger
baggage
```

and C contains:

```text
invoice
tweet
budget
network
permission
```

A TF-IDF classifier can become good at recognizing patterns inside domains it has already seen.

It doesn't inherently understand that:

```text
"uses a ZIP code the user never provided"
```

and

```text
"uses an account ID that was never established"
```

are semantically similar errors.

That's where a semantic representation could potentially help.

---

# 7. What should stay frozen now?

I would stop modifying these parts:

```text
✓ Target:
  -1 / 0 / +1

✓ Grouping:
  trajectory-level

✓ Primary metric:
  Macro F1

✓ Secondary metrics:
  -1 F1
  -1 recall

✓ Context:
  2 previous steps

✓ Feature families:
  structural
  current text
  context text

✓ Validation:
  grouped CV + leave-one-dataset-out
```

These form your experimental framework.

Now you can swap **models/representations** while keeping everything else constant.

That's important scientifically because then:

```text
Model A = 0.60
Model B = 0.66
```

actually means something. You're not changing the dataset, context, splitting and features simultaneously.

---

# 8. What models should come next?

I'd use a small progression rather than trying 15 algorithms.

### Model 0 — Dummy

Already done.

```text
Macro F1 ≈ 0.267
```

### Model 1 — Logistic Regression + TF-IDF

Already done.

```text
Mixed CV ≈ 0.602

Cross-domain:
0.305
0.407
0.411
```

This is your **classical baseline**.

### Model 2 — LinearSVC + TF-IDF

Do this next.

Keep exactly:

```text
context = 2
same structural features
same TF-IDF
same splits
```

Only replace:

```python
LogisticRegression(...)
```

with something like:

```python
from sklearn.svm import LinearSVC

LinearSVC(
    C=1.0,
    random_state=42,
)
```

Then repeat:

```text
5-fold grouped CV

AND

A+B → C
A+C → B
B+C → A
```

This tells you whether the limitation is partly the classifier.

---

# 9. Then semantic representations

This is where I think your project becomes more interesting.

Instead of:

```text
text
 ↓
TF-IDF
 ↓
sparse lexical vector
```

try:

```text
current step ──────→ semantic encoder ──→ embedding
                                             │
context ───────────→ semantic encoder ──→ embedding
                                             │
structural features ─────────────────────────┤
                                             ↓
                                         classifier
```

You could initially **freeze the embedding model** and train a simple classifier on top.

That gives you a clean comparison:

```text
TF-IDF + Logistic Regression

vs

Semantic embeddings + Logistic Regression
```

You don't need to fine-tune a transformer immediately.

---

# 10. Your most important future table

Ultimately I'd aim to produce this:

| Model                    | Mixed Grouped CV |     A+B→C |     A+C→B |     B+C→A |
| ------------------------ | ---------------: | --------: | --------: | --------: |
| Dummy                    |            0.267 |         ? |         ? |         ? |
| Logistic + TF-IDF        |        **0.602** | **0.305** | **0.407** | **0.411** |
| LinearSVC + TF-IDF       |                ? |         ? |         ? |         ? |
| Semantic embeddings + LR |                ? |         ? |         ? |         ? |
| Transformer              |                ? |         ? |         ? |         ? |

And separately retain your ablation table.

Those two tables answer two different research questions:

**Ablation:**

> What information is needed to detect an agent error?

Answer so far: current action is strongest, but local context and structural information provide complementary signal.

**Cross-dataset:**

> Does the detector generalize to unseen agent environments?

Answer so far: only partially; there is a substantial domain-generalization gap.

---

## So where your project stands

I'd summarize your findings right now as:

> **Step-level agent failures are predictable above a naive baseline using lightweight classical ML. Current-step text provides the strongest individual signal, while recent conversational context and structural trajectory features provide complementary gains. Short local context performs better than long trajectory histories. However, performance drops substantially when transferring to an unseen benchmark, indicating that TF-IDF-based models rely heavily on benchmark-specific lexical and structural patterns.**

That's a much stronger project story than simply saying *“my classifier gets 60% Macro F1.”*

Your **immediate next experiment is LinearSVC using exactly the same data, window=2, features, grouped folds, and cross-dataset tests**. After that, the priority should shift to semantic embeddings and whether they reduce the cross-dataset generalization gap.
